<a href="https://colab.research.google.com/github/GabrielJ07/ConfiguratorAgent/blob/main/Copy_of_Convert_OpenAI_to_Markdown.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import os
from datetime import datetime

def convert_openai_to_md(input_file, output_file):
    """Parses OpenAI's node-based conversations.json and converts it to NotebookLM-friendly Markdown."""

    if not os.path.exists(input_file):
        print(f"Error: Could not find '{input_file}'. Make sure it is in the same folder as this script.")
        return

    print(f"Reading {input_file}...")
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # NotebookLM 500k word limit safeguard (using 400k for a safe buffer)
    MAX_WORDS_PER_FILE = 400000
    base_name, ext = os.path.splitext(output_file)
    file_index = 1

    current_out_file = f"{base_name}_Part{file_index}{ext}"
    out_f = open(current_out_file, 'w', encoding='utf-8')
    out_f.write(f"# Circuittelligence - ChatGPT Activity History (Part {file_index})\n\n")
    print(f"Writing to {current_out_file}...")

    current_word_count = 0
    convo_count = 0
    message_count = 0

    for convo in data:
        convo_content = ""
        title = convo.get('title', 'Untitled Conversation')
        create_time = convo.get('create_time', 0)

        # Format the timestamp
        if create_time:
            dt = datetime.fromtimestamp(create_time)
            formatted_time = dt.strftime("%B %d, %Y at %I:%M %p")
        else:
            formatted_time = "Unknown Date"

        convo_content += f"## Conversation: {title}\n"
        convo_content += f"**Started:** {formatted_time}\n\n"

        mapping = convo.get('mapping', {})
        current_node = convo.get('current_node')

        # OpenAI uses a tree. To get the actual linear chat you saw, we start at the
        # last node (current_node) and walk backward through the 'parent' references.
        path = []
        curr = current_node
        while curr and curr in mapping:
            node = mapping[curr]
            path.append(node)
            curr = node.get('parent')

        # Reverse the path so it reads chronologically from first to last
        path.reverse()

        for node in path:
            message = node.get('message')
            if not message:
                continue

            author_role = message.get('author', {}).get('role', 'unknown')

            # We only want user prompts and assistant (ChatGPT) responses, skip system prompts
            if author_role not in ['user', 'assistant']:
                continue

            display_sender = "User Prompt" if author_role == 'user' else "ChatGPT Response"

            content = message.get('content', {})
            content_type = content.get('content_type', '')
            parts = content.get('parts', [])

            # Extract text (skipping non-text elements like tool calls if present)
            text = ""
            if content_type == 'text':
                text = "".join([str(p) for p in parts if isinstance(p, str)])

            text = text.strip()

            if text:
                convo_content += f"**{display_sender}:**\n{text}\n\n"
                message_count += 1

        convo_content += "---\n\n"

        # Word count safety check
        words_in_convo = len(convo_content.split())

        if current_word_count + words_in_convo > MAX_WORDS_PER_FILE and current_word_count > 0:
            out_f.close()
            file_index += 1
            current_out_file = f"{base_name}_Part{file_index}{ext}"
            out_f = open(current_out_file, 'w', encoding='utf-8')
            out_f.write(f"# Circuittelligence - ChatGPT Activity History (Part {file_index})\n\n")
            print(f"File limit reached. Continuing in {current_out_file}...")
            current_word_count = 0

        out_f.write(convo_content)
        current_word_count += words_in_convo
        convo_count += 1

    out_f.close()

    print(f"Success! Converted {convo_count} conversations ({message_count} messages) across {file_index} file(s).")
    print("You can now upload these files directly into NotebookLM.")

if __name__ == "__main__":
    # Point this to the specific OpenAI file
    INPUT_FILENAME = 'conversations.json'
    OUTPUT_FILENAME = 'Circuittelligence_ChatGPT_Activity.md'

    convert_openai_to_md(INPUT_FILENAME, OUTPUT_FILENAME)